In [0]:
df = spark.sql('select * from `ete-catalog`.silver.orders')

df.limit(3).display()

### **Reading dfs from gold schema**

In [0]:
df_cust = spark.sql('select DimCustomerKey as dim_cust_key, customer_id  as cust_id from `ete-catalog`.gold.dimcustomers')

df_pro = spark.sql('select product_id as dim_pro_key, product_id as pro_id from `ete-catalog`.gold.dim_products')

### **Joining for fact table**

In [0]:
df_fact = df.join(df_cust,df['customer_id'] == df_cust['cust_id'], how = 'left').join(df_pro,df['product_id'] == df_cust['pro_id'], how = 'left')

df_fact = df_fact.drop('customer_id','product_id','cust_id','pro_id')

df_fact.limit(3).display()

### **Upsert Command**

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('`ete-catalog`.gold.fact_orders') :
    
    obj = DeltaTable.forPath(spark, 'abfss://gold@etehub.dfs.core.windows.net/fact_orders')

    obj.alias('tar').merge(df_fact.alias('src'), "tar.dim_cust_key = src.dim_cust_key AND tar.dim_pro_key = src.dim_pro_key")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()


else :
    df_fact.write.format('delta')\
        .option('path', 'abfss://gold@etehub.dfs.core.windows.net/fact_orders')\
        .saveAsTable('`ete-catalog`.gold.fact_orders')